# Projeto Original — Análise de Sentimentos com Embeddings

**Ciclo de divergência: `MODEL`.**

O projeto FMF resolve a classificação binária de sentimento em reviews do IMDB usando **TF-IDF + algoritmos clássicos de ML**. Aqui mantemos exatamente o mesmo problema, os mesmos dados e o mesmo classificador final, e trocamos apenas a **representação textual** — que é o nosso ponto de divergência.

Comparamos 4 representações, da mais lexical à mais semântica:

1. **TF-IDF** — reproduz o FMF (linha de base).
2. **BM25** — baseline lexical forte (TF-IDF melhorado: saturação de termo + normalização por comprimento).
3. **BGE-large-en-v1.5** — embedding denso especialista em inglês, SOTA, mas trunca em 512 tokens.
4. **BGE-M3** — embedding denso de **contexto longo (8192 tokens)**: lê a review inteira sem truncar.

Todas as representações alimentam os **mesmos classificadores** (SVM linear e Regressão Logística), isolando a variável "representação textual".

> **Pergunta de pesquisa do ciclo:** ler a review inteira (BGE-M3) compensa, ou um encoder inglês mais forte vence mesmo truncando? E os embeddings densos realmente superam um bom método lexical (BM25) num dataset pequeno?

## Dependências

Execute a célula abaixo uma vez para instalar as bibliotecas necessárias (descomente se ainda não tiver instalado).

In [1]:
!pip install pandas scikit-learn imbalanced-learn matplotlib seaborn sentence-transformers torch

  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 19.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 24.0 MB/s  0:00:01m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 43.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 27.3 MB/s  0:00:00eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
   ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/532.3 MB 19.2 MB/s eta 0:00:22

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
ERRO

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=13)
plt.rc('ytick', labelsize=13)
plt.rc('legend', fontsize=13)
plt.rc('font', size=13)

RANDOM_STATE = 42

# GET — Obtenção dos dados

Usamos o **mesmo dataset e a mesma amostragem do FMF**, garantindo que os conjuntos de treino e teste sejam idênticos (mesmo `random_state`). Assim, qualquer diferença de desempenho vem da representação textual, não dos dados.

> O arquivo `IMDB Dataset.csv` (50k reviews) deve ser baixado do [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) e colocado na raiz do repositório ou na pasta `FMF/`. A célula abaixo procura em vários caminhos.

In [3]:
import os

CANDIDATE_PATHS = [
    'IMDB Dataset.csv',
    '../IMDB Dataset.csv',
    '../FMF/IMDB Dataset.csv',
    'FMF/IMDB Dataset.csv',
]

csv_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        'IMDB Dataset.csv não encontrado. Baixe do Kaggle e coloque na raiz do repositório.'
    )

df_review = pd.read_csv(csv_path)
print(f'Carregado de: {csv_path}')
df_review.head()

Carregado de: ../IMDB Dataset.csv


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Mesma amostragem desbalanceada do FMF: 9000 positivos + 1000 negativos
df_positive = df_review[df_review['sentiment'] == 'positive'][:9000]
df_negative = df_review[df_review['sentiment'] == 'negative'][:1000]
df_review_imb = pd.concat([df_positive, df_negative])

# Balanceamento por undersampling (igual ao FMF)
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=0)
df_review_bal, df_review_bal['sentiment'] = rus.fit_resample(
    df_review_imb[['review']], df_review_imb['sentiment']
)
df_review_bal.value_counts('sentiment')

sentiment
negative    1000
positive    1000
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df_review_bal, test_size=0.33, random_state=RANDOM_STATE)
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

print('Treino:', train_x.shape[0], '| Teste:', test_x.shape[0])
print(train_y.value_counts())

Treino: 1340 | Teste: 660
sentiment
negative    675
positive    665
Name: count, dtype: int64


In [6]:
# Estatística de comprimento das reviews — motiva o uso de modelos de contexto longo
lengths = train_x.str.split().str.len()
print(f'Palavras por review (treino): média={lengths.mean():.0f}, '
      f'mediana={lengths.median():.0f}, p95={lengths.quantile(0.95):.0f}, máx={lengths.max()}')
print(f'% de reviews com >512 palavras: {(lengths > 512).mean() * 100:.1f}%')

Palavras por review (treino): média=230, mediana=173, p95=570, máx=1011
% de reviews com >512 palavras: 7.5%


# Função de avaliação comum

Para uma comparação justa, todas as representações usam o mesmo procedimento: treinar **SVM linear** e **Regressão Logística** sobre os vetores e avaliar com acurácia, F1 e matriz de confusão. Os resultados são acumulados em `RESULTS`.

In [7]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

LABELS = ['positive', 'negative']
RESULTS = []


def evaluate_representation(name, X_train, X_test, y_train=train_y, y_test=test_y, verbose=True):
    """Treina SVM linear e Regressão Logística e registra as métricas."""
    models = {
        'SVM (linear)': SVC(kernel='linear', C=1),
        'LogReg': LogisticRegression(max_iter=1000),
    }
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        f1_pos, f1_neg = f1_score(y_test, pred, labels=LABELS, average=None)
        f1_macro = f1_score(y_test, pred, labels=LABELS, average='macro')
        RESULTS.append({
            'Representação': name,
            'Classificador': model_name,
            'Acurácia': acc,
            'F1 (pos)': f1_pos,
            'F1 (neg)': f1_neg,
            'F1 macro': f1_macro,
        })
        if verbose:
            print(f'\n=== {name} — {model_name} ===')
            print(f'Acurácia: {acc:.4f} | F1 macro: {f1_macro:.4f}')
            print(classification_report(y_test, pred, labels=LABELS))
    return models

# Representação 1 — TF-IDF (reprodução do FMF)

Linha de base: o mesmo `TfidfVectorizer` do projeto FMF.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_train = tfidf.fit_transform(train_x)
tfidf_test = tfidf.transform(test_x)
print('Matriz TF-IDF:', tfidf_train.shape)

evaluate_representation('TF-IDF', tfidf_train, tfidf_test)

Matriz TF-IDF: (1340, 20625)

=== TF-IDF — SVM (linear) ===
Acurácia: 0.8409 | F1 macro: 0.8407
              precision    recall  f1-score   support

    positive       0.83      0.87      0.85       335
    negative       0.85      0.82      0.83       325

    accuracy                           0.84       660
   macro avg       0.84      0.84      0.84       660
weighted avg       0.84      0.84      0.84       660


=== TF-IDF — LogReg ===
Acurácia: 0.8303 | F1 macro: 0.8299
              precision    recall  f1-score   support

    positive       0.81      0.87      0.84       335
    negative       0.85      0.79      0.82       325

    accuracy                           0.83       660
   macro avg       0.83      0.83      0.83       660
weighted avg       0.83      0.83      0.83       660



{'SVM (linear)': SVC(C=1, kernel='linear'),
 'LogReg': LogisticRegression(max_iter=1000)}

# Representação 2 — BM25 (baseline lexical forte)

BM25 é uma evolução do TF-IDF: além da frequência inversa de documento (IDF), aplica **saturação de termo** (com o parâmetro `k1`, frequências altas param de crescer linearmente) e **normalização por comprimento do documento** (com `b`). É o padrão de mercado em busca textual (ex.: Lucene/Elasticsearch).

Como o `scikit-learn` não traz BM25, implementamos um vetorizador BM25 compatível com a API `fit`/`transform`, reaproveitando a contagem de termos do `CountVectorizer`. O resultado é uma matriz esparsa documento-termo pronta para os mesmos classificadores.

In [9]:
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.base import BaseEstimator, TransformerMixin


class BM25Vectorizer(BaseEstimator, TransformerMixin):
    """Vetorizador BM25 (variante Lucene/Okapi com IDF não-negativo).

    Aprende vocabulário, IDF e comprimento médio dos documentos no fit;
    no transform produz a matriz esparsa de pesos BM25 por termo.
    """

    def __init__(self, k1=1.5, b=0.75, stop_words='english'):
        self.k1 = k1
        self.b = b
        self.stop_words = stop_words

    def fit(self, raw_documents, y=None):
        self.cv_ = CountVectorizer(stop_words=self.stop_words)
        tf = self.cv_.fit_transform(raw_documents)            # contagem de termos
        n_docs = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1])    # nº de docs por termo
        # IDF estilo Lucene: sempre >= 0
        self.idf_ = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
        self.avgdl_ = tf.sum(axis=1).mean()
        return self

    def transform(self, raw_documents):
        tf = self.cv_.transform(raw_documents).tocsr()
        doc_len = np.asarray(tf.sum(axis=1)).ravel()
        tf = tf.tocoo()
        # denominador da fórmula BM25, por elemento não-nulo
        denom = tf.data + self.k1 * (1 - self.b + self.b * doc_len[tf.row] / self.avgdl_)
        scores = self.idf_[tf.col] * (tf.data * (self.k1 + 1)) / denom
        return sp.csr_matrix((scores, (tf.row, tf.col)), shape=tf.shape)

    def get_feature_names_out(self, input_features=None):
        return self.cv_.get_feature_names_out(input_features)

In [10]:
bm25 = BM25Vectorizer()
bm25_train = bm25.fit_transform(train_x)
bm25_test = bm25.transform(test_x)
print('Matriz BM25:', bm25_train.shape)

evaluate_representation('BM25', bm25_train, bm25_test)

Matriz BM25: (1340, 20625)

=== BM25 — SVM (linear) ===
Acurácia: 0.8455 | F1 macro: 0.8453
              precision    recall  f1-score   support

    positive       0.84      0.87      0.85       335
    negative       0.86      0.82      0.84       325

    accuracy                           0.85       660
   macro avg       0.85      0.85      0.85       660
weighted avg       0.85      0.85      0.85       660


=== BM25 — LogReg ===
Acurácia: 0.8515 | F1 macro: 0.8513
              precision    recall  f1-score   support

    positive       0.84      0.88      0.86       335
    negative       0.87      0.82      0.85       325

    accuracy                           0.85       660
   macro avg       0.85      0.85      0.85       660
weighted avg       0.85      0.85      0.85       660



{'SVM (linear)': SVC(C=1, kernel='linear'),
 'LogReg': LogisticRegression(max_iter=1000)}

# Embeddings densos

A partir daqui trocamos representações esparsas (palavra-a-palavra) por **embeddings densos**: vetores contínuos que capturam significado semântico, gerados por modelos pré-treinados. Carregamos os modelos com `sentence-transformers` e usamos GPU se disponível.

> O download dos modelos (centenas de MB a ~2 GB) ocorre na primeira execução. Em CPU, codificar ~2000 textos leva alguns minutos.

In [11]:
from sentence_transformers import SentenceTransformer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


def embed(model_name, texts, **kwargs):
    model = SentenceTransformer(model_name, device=DEVICE)
    emb = model.encode(
        list(texts),
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
        **kwargs,
    )
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return emb

/home/felipe/modeling_sentiment_analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


# Representação 3 — BGE-large-en-v1.5

Embedding denso **especialista em inglês**, no topo do MTEB para tarefas de classificação. Janela de contexto de **512 tokens** — reviews mais longas são truncadas, perdendo o final do texto.

In [12]:
bge_train = embed('BAAI/bge-large-en-v1.5', train_x)
bge_test = embed('BAAI/bge-large-en-v1.5', test_x)
print('Embeddings BGE-large-en:', bge_train.shape)

evaluate_representation('BGE-large-en-v1.5', bge_train, bge_test)

Batches:  14%|█▍        | 6/42 [02:16<13:39, 22.76s/it]


KeyboardInterrupt: 

# Representação 4 — BGE-M3

Embedding denso multilíngue com **contexto longo (8192 tokens)**: consegue ler a review **inteira** sem truncar. Este é o diferencial técnico central do nosso ciclo de divergência.

In [ ]:
bgem3_train = embed('BAAI/bge-m3', train_x)
bgem3_test = embed('BAAI/bge-m3', test_x)
print('Embeddings BGE-M3:', bgem3_train.shape)

evaluate_representation('BGE-M3', bgem3_train, bgem3_test)

# COMMUNICATE — Comparação dos resultados

Consolidamos todas as combinações representação × classificador numa única tabela e gráfico.

In [ ]:
df_results = pd.DataFrame(RESULTS)
df_results = df_results.sort_values('F1 macro', ascending=False).reset_index(drop=True)
df_results.style.format({
    'Acurácia': '{:.3f}', 'F1 (pos)': '{:.3f}', 'F1 (neg)': '{:.3f}', 'F1 macro': '{:.3f}'
}).background_gradient(subset=['F1 macro'], cmap='Greens')

In [ ]:
# Melhor classificador por representação
best = df_results.loc[df_results.groupby('Representação')['F1 macro'].idxmax()]
best = best.sort_values('F1 macro', ascending=False)

plt.figure(figsize=(9, 5), tight_layout=True)
colors = sns.color_palette('deep')
bars = plt.bar(best['Representação'], best['F1 macro'], color=colors[:len(best)])
plt.ylabel('F1 macro (melhor classificador)')
plt.title('Representação textual × desempenho')
plt.ylim(0, 1)
plt.xticks(rotation=15)
for bar, val in zip(bars, best['F1 macro']):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.3f}', ha='center')
plt.savefig('comparacao_representacoes.png', dpi=120)
plt.show()

## Conclusões

*(Preencher após a execução, com base na tabela e no gráfico acima.)*

Pontos a discutir no relatório:

- **Lexical vs. semântico:** os embeddings densos (BGE) superaram os métodos lexicais (TF-IDF, BM25) neste dataset pequeno (~1.340 exemplos de treino)? Em corpora pequenos é comum que baselines lexicais com SVM linear sejam muito competitivos.
- **BM25 vs. TF-IDF:** a saturação de termo e a normalização por comprimento do BM25 trouxeram ganho sobre o TF-IDF do FMF?
- **Contexto longo (BGE-M3) vs. contexto curto (BGE-large-en):** ler a review inteira compensou, ou o encoder inglês mais forte venceu mesmo truncando em 512 tokens? Relacionar com a % de reviews longas medida na etapa GET.
- **Custo × benefício:** os embeddings exigem download de modelos grandes e mais tempo de inferência. O ganho de desempenho justifica esse custo frente ao TF-IDF/BM25?